# 03_HT
---

![](img/ht.png)

In [22]:
# se limpia memoria
rm(list=ls())
gc()

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,2390701,127.7,4525722,241.7,NA,4525722,241.7
Vcells,4258552,32.5,10146329,77.5,16384,10146327,77.5


In [23]:
library(yaml)
require("data.table")
require("primes")
require("lightgbm") #Optimización bayesiana
require("DiceKriging")
require("mlrMBO")
require("rlist")

In [24]:
setwd("/Users/ppando/Materias/cs-datos/especializacion/ciencia-datos/exp/")

In [25]:
PARAMS <- read_yaml("03_HT_prxs_01a.yml")

In [26]:
# se graba a un archivo los componentes de lista
# para el primer registro, escribe antes los titulos

loguear  <- function(reg, arch=NA, folder="./exp/", ext=".txt", verbose=TRUE) {
  archivo  <- arch
  if (is.na(arch))
      archivo  <- paste0(folder, substitute(reg), ext)
  
  if (!file.exists(archivo)) {
    linea  <- paste0("fecha\t", paste(list.names(reg), collapse="\t" ), "\n")
    cat(linea, file = archivo)
  }
  
  linea  <- paste0(format(Sys.time(), "%Y%m%d %H%M%S"),  "\t", gsub( ", ", "\t", toString(reg)),  "\n")
  cat(linea, file = archivo, append = TRUE)
  if(verbose)
      cat(linea)   # se imprime por pantalla
}

In [27]:
parametrizar  <- function(lparam) {
  param_fijos  <- copy(lparam)
  hs  <- list()

  for (param in names(lparam)) {

    if (length(lparam[[param]]) > 1) {
      desde  <- as.numeric(lparam[[param]][[1]])
      hasta  <- as.numeric(lparam[[param]][[2]])

      if (length(lparam[[param]]) == 2) {
         hs  <- append(hs, list(makeNumericParam(param, lower= desde, upper= hasta)))
      } else {
         hs  <- append(hs, list(makeIntegerParam(param, lower= desde, upper= hasta)))
      }

      param_fijos[[param]] <- NULL  
    }
  }

  return(list("param_fijos" = param_fijos, "paramSet" = hs))
}

In [28]:
# particionar un dataset en forma estratificada
particionar  <- function(data,  division, agrupa="",  campo="fold", start=1, seed=NA) {
  if (!is.na(seed))
      set.seed(seed)

  bloque  <- unlist(mapply(function(x,y) {rep(y, x)}, division, seq(from=start, length.out=length(division))))  

  data[ , (campo) :=  sample(rep(bloque, ceiling(.N/length(bloque))))[1:.N], by= agrupa]
}

In [29]:
EstimarGanancia_lightgbm  <- function(x) {
  gc()
  GLOBAL_iteracion  <<- GLOBAL_iteracion + 1

  param_completo  <- c(param_fijos, x)

  param_completo$num_iterations         <- ifelse(param_fijos$boosting== "dart", 999, 99999)  #un numero muy grande
  param_completo$early_stopping_rounds  <- as.integer(200 + 4/param_completo$learning_rate)

  # vprob_optima  <<- c()
  set.seed(param_completo$seed)
  modelo_train  <- lgb.train( data= dtrain,
                              valids= list(valid = dvalidate),
                              param=  param_completo,
                              verbose= -100 )


  #aplico el modelo a testing y calculo la ganancia
  prediccion  <- predict( modelo_train, 
                          data.matrix( dataset_test[ , campos_buenos, with=FALSE]))

  tbl  <- dataset_test[ , c(PARAMS$param$const$campo_clase),with=F]
  tbl[ , pred := prediccion]

  gc()

  ganancia_test_normalizada  <- unlist(modelo_train$record_evals$valid[[PARAMS$param$lightgbm$metric]]$eval)[ modelo_train$best_iter ]
  
  rm(tbl)
  gc()
  
  #voy grabando las mejores column importance
  if (ganancia_test_normalizada <  GLOBAL_ganancia) {
    GLOBAL_ganancia  <<- ganancia_test_normalizada
    tb_importancia    <- as.data.table(lgb.importance(modelo_train))

    fwrite( tb_importancia,
            file= paste0(PARAMS$param$files$output$importancia, GLOBAL_iteracion, ".txt"),
            sep= "\t" )
  }

  #logueo final
  ds  <- list("cols"= ncol(dtrain),  "rows"= nrow(dtrain))
  xx  <- c(ds, copy(param_completo))

  xx$early_stopping_rounds  <- NULL
  xx$num_iterations  <- modelo_train$best_iter
  xx$ganancia  <- ganancia_test_normalizada
  xx$iteracion_bayesiana  <- GLOBAL_iteracion

  loguear(xx, arch = PARAMS$param$files$output$BOlog)

  return (ganancia_test_normalizada)
}

## Main

In [ ]:
set.seed(PARAMS$param$semilla) 

#se carga el dataset que tiene la Training Strategy
nom_arch <- PARAMS$param$files$input$dentrada 
dataset  <- fread(nom_arch)

# los campos que se pueden utilizar para la prediccion
campos_buenos <- setdiff(copy(colnames(dataset)), c(PARAMS$param$const$campo_clase, "part_train","part_validate","part_test"))

# la particion de train siempre va
dtrain  <- lgb.Dataset(data =  data.matrix(dataset[part_train==1, campos_buenos, with=FALSE]),
                       label = dataset[part_train==1][[PARAMS$param$const$campo_clase]],
                       free_raw_data = FALSE)


# se calcula validation y testing, segun corresponda
if (PARAMS$param$crossvalidation == FALSE) {
  if (PARAMS$param$validate == TRUE) {
    dvalidate  <- lgb.Dataset(data =  data.matrix(dataset[part_validate==1, campos_buenos, with=FALSE]),
                              label = dataset[part_validate==1][[PARAMS$param$const$campo_clase]],
                              free_raw_data = FALSE)

    dataset_test  <- dataset[part_test== 1]
    test_multiplicador  <- 1
  } else {
    # se divide en mitades los datos de testing
    particionar( dataset, 
                 division= c(1,1),
                 agrupa= c("part_test", PARAMS$param$const$campo_periodo), 
                 seed= PARAMS$param$semilla,
                 campo= "fold_test"
                )

    # fold_test==1  lo tomo para validation
    dvalidate  <- lgb.Dataset( data=  data.matrix(dataset[ part_test==1 & fold_test==1, campos_buenos, with=FALSE]),
                               label= dataset[part_test==1 & fold_test==1, PARAMS$param$const$campo_clase],
                               free_raw_data= FALSE
                             )

    dataset_test  <- dataset[part_test==1 & fold_test==2, ]
    test_multiplicador  <- 2
  }

}


rm(dataset)
gc()


# se prepara todo para la Bayesian Optimization -------------------------------------
hiperparametros <- PARAMS$param[[PARAMS$param$algoritmo]]
apertura  <- parametrizar(hiperparametros)
param_fijos  <-  apertura$param_fijos


#si ya existe el archivo log, traigo hasta donde procese
if (file.exists(PARAMS$param$files$output$BOlog)) {
  tabla_log  <- fread(PARAMS$param$files$output$BOlog)
  GLOBAL_iteracion  <- nrow(tabla_log)
  GLOBAL_ganancia   <- tabla_log[ , min(ganancia)]
  rm(tabla_log)
} else  {
  GLOBAL_iteracion  <- 0
  GLOBAL_ganancia   <- Inf
}

# aqui comienza la configuracion de mlrMBO
if (PARAMS$param$crossvalidation) {
  funcion_optimizar  <- EstimarGanancia_lightgbmCV
} else {
  funcion_optimizar  <- EstimarGanancia_lightgbm
}

configureMlr(show.learner.output= FALSE)

# se configura la busqueda bayesiana, los hiperparametros que se van a optimizar
obj.fun  <- makeSingleObjectiveFunction(
              fn =      funcion_optimizar, #la funcion que voy a maximizar
              minimize= PARAMS$param$BO$minimize,   #estoy Maximizando la ganancia
              noisy=    PARAMS$param$BO$noisy,
              par.set=  makeParamSet( params= apertura$paramSet ),     #definido al comienzo del programa
              has.simple.signature = PARAMS$param$BO$has.simple.signature   #paso los parametros en una lista
             )

#archivo donde se graba y cada cuantos segundos
ctrl  <- makeMBOControl(save.on.disk.at.time= PARAMS$param$BO$save.on.disk.at.time, save.file.path= PARAMS$param$files$output$BObin )  
                         
ctrl  <- setMBOControlTermination(ctrl, iters= PARAMS$param$BO$iterations)   #cantidad de iteraciones
                                   
ctrl  <- setMBOControlInfill(ctrl, crit= makeMBOInfillCritEI())

#establezco la funcion que busca el maximo
surr.km  <- makeLearner("regr.km",
                        predict.type= "se",
                        covtype= "matern3_2",
                        control= list(trace= TRUE))

# aqui se inicia la optimizacion bayesiana
if (!file.exists(PARAMS$param$files$output$BObin)) {
    run  <- mbo(obj.fun, learner= surr.km, control= ctrl)
} else {
    # si ya existe el archivo RDATA, se debe continuar desde el punto hasta donde llegue
    # usado para cuando se corta la virtual machine
    run  <- mboContinue( PARAMS$param$files$output$BObin)   # se retoma en caso que ya exista
}

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,2491083,133.1,4525722,241.7,NA,4525722,241.7
Vcells,8090994,61.8,21308951,162.6,16384,18972579,144.8


Computing y column(s) for design. Not provided.



20251123 172620	32	98054	0	0	0	1	1	1	-1	1023	999983	FALSE	0.1	50	0.5	rmse	TRUE	regression	TRUE	TRUE	FALSE	gbdt	0	-100	-100	0.0684417041653069	0.667025725623535	3777	54796	1	2.16784376388978	1
20251123 172622	32	98054	0	0	0	1	1	1	-1	1023	999983	FALSE	0.1	50	0.5	rmse	TRUE	regression	TRUE	TRUE	FALSE	gbdt	0	-100	-100	0.191535269288579	0.554696707922267	13378	63812	1	2.16784376388978	2
20251123 172643	32	98054	0	0	0	1	1	1	-1	1023	999983	FALSE	0.1	50	0.5	rmse	TRUE	regression	TRUE	TRUE	FALSE	gbdt	0	-100	-100	0.0851192367571639	0.375601522153011	12222	46501	1763	2.13675665289513	3
20251123 172719	32	98054	0	0	0	1	1	1	-1	1023	999983	FALSE	0.1	50	0.5	rmse	TRUE	regression	TRUE	TRUE	FALSE	gbdt	0	-100	-100	0.258500224731397	0.547033663497132	18705	26423	2880	1.84290797442822	4
20251123 172832	32	98054	0	0	0	1	1	1	-1	1023	999983	FALSE	0.1	50	0.5	rmse	TRUE	regression	TRUE	TRUE	FALSE	gbdt	0	-100	-100	0.105658529841457	0.820036890696792	6010	40616	15591	2.06129061767349	5
20251123 172835	32	98054	0	0	0

[mbo] 0: learning_rate=0.0684; feature_fraction=0.667; num_leaves=3777; min_data_in_leaf=54796 : y = 2.17 : 1.1 secs : initdesign

[mbo] 0: learning_rate=0.192; feature_fraction=0.555; num_leaves=13378; min_data_in_leaf=63812 : y = 2.17 : 2.4 secs : initdesign

[mbo] 0: learning_rate=0.0851; feature_fraction=0.376; num_leaves=12222; min_data_in_leaf=46501 : y = 2.14 : 20.2 secs : initdesign

[mbo] 0: learning_rate=0.259; feature_fraction=0.547; num_leaves=18705; min_data_in_leaf=26423 : y = 1.84 : 36.3 secs : initdesign

[mbo] 0: learning_rate=0.106; feature_fraction=0.82; num_leaves=6010; min_data_in_leaf=40616 : y = 2.06 : 73.4 secs : initdesign

[mbo] 0: learning_rate=0.117; feature_fraction=0.173; num_leaves=187; min_data_in_leaf=49064 : y = 2.17 : 2.5 secs : initdesign

[mbo] 0: learning_rate=0.17; feature_fraction=0.479; num_leaves=9871; min_data_in_leaf=11807 : y = 1.81 : 18.7 secs : initdesign

[mbo] 0: learning_rate=0.14; feature_fraction=0.246; num_leaves=17449; min_data_in_l

20251123 173614	32	98054	0	0	0	1	1	1	-1	1023	999983	FALSE	0.1	50	0.5	rmse	TRUE	regression	TRUE	TRUE	FALSE	gbdt	0	-100	-100	0.0329569284702933	0.604969568810164	19996	16082	9816	1.83769260940135	17


...

[mbo] 37: learning_rate=0.0202; feature_fraction=0.371; num_leaves=7082; min_data_in_leaf=50 : y = 1.75 : 154.4 secs : infill_ei



20251123 204035	32	98054	0	0	0	1	1	1	-1	1023	999983	FALSE	0.1	50	0.5	rmse	TRUE	regression	TRUE	TRUE	FALSE	gbdt	0	-100	-100	0.0200886714382822	0.843362074965486	10960	0	87	1.84148024749431	54


[mbo] 38: learning_rate=0.0201; feature_fraction=0.843; num_leaves=10960; min_data_in_leaf=0 : y = 1.84 : 328.0 secs : infill_ei



20251123 204054	32	98054	0	0	0	1	1	1	-1	1023	999983	FALSE	0.1	50	0.5	rmse	TRUE	regression	TRUE	TRUE	FALSE	gbdt	0	-100	-100	0.0302579649463154	0.136510104026465	3420	13	172	1.76705952192567	55


[mbo] 39: learning_rate=0.0303; feature_fraction=0.137; num_leaves=3420; min_data_in_leaf=13 : y = 1.77 : 19.0 secs : infill_ei

